In [2]:
from pathlib import Path
import json

# Our directories
sections_dir = Path("temporary_sections")
responses_dir = Path("../../data/test/raw_api_responses")

# Get all temporary audio sections
audio_files = sorted(sections_dir.glob("*.wav"))

# Get names of sections already transcribed
completed = {
    file.stem
    for file in responses_dir.glob("section_*.response")
}

# Find sections that still need transcription
pending = [
    file for file in audio_files
    if file.stem not in completed
]

print("Total audio sections:", len(audio_files))
print("Already transcribed:", len(audio_files) - len(pending))
print("Still need transcription:", len(pending))

print("\nFirst 5 pending sections:")
for file in pending[:5]:
    print(file.name)

Total audio sections: 10
Already transcribed: 10
Still need transcription: 0

First 5 pending sections:


In [3]:
import subprocess

original_audio = Path("../../data/test/test_audio.mp3")

def get_duration(path):
    result = subprocess.run(
        [
            "ffprobe",
            "-v", "error",
            "-show_entries", "format=duration",
            "-of", "default=noprint_wrappers=1:nokey=1",
            str(path)
        ],
        capture_output=True,
        text=True,
        check=True
    )

    return float(result.stdout.strip())

if original_audio.exists():
    duration = get_duration(original_audio)

    print("Original duration:", round(duration, 2), "seconds")
    print("Original duration:", round(duration / 60, 2), "minutes")
else:
    print("Original audio not found:", original_audio)

Original duration: 591.73 seconds
Original duration: 9.86 minutes


In [4]:
metadata_path = Path("temporary_sections.json")

metadata = json.loads(
    metadata_path.read_text(encoding="utf-8")
)

print("Metadata type:", type(metadata).__name__)

if isinstance(metadata, list):
    print("Number of entries:", len(metadata))
    print(json.dumps(metadata[:2], indent=2, ensure_ascii=False))

elif isinstance(metadata, dict):
    print("Top-level keys:", list(metadata.keys()))
    print(
        json.dumps(metadata, indent=2, ensure_ascii=False)[:2000]
    )

Metadata type: list
Number of entries: 10
[
  {
    "filename": "section_000.wav",
    "start": 0,
    "end": 59.9
  },
  {
    "filename": "section_001.wav",
    "start": 58.9,
    "end": 122.55
  }
]


In [5]:
import re
from rapidfuzz import fuzz

# Read the entire book
book = Path("../../data/test/book.txt").read_text(
    encoding="utf-8"
)

# Normalize text for matching
def normalize(text):
    text = text.lower()

    for mark in ["‘", "’", "ʻ", "ʼ", "`"]:
        text = text.replace(mark, "'")

    text = re.sub(r"[^\w\s']", " ", text)
    return " ".join(text.split())


book_words = normalize(book).split()

# Read all existing transcription responses
response_files = sorted(
    responses_dir.glob("section_*.response")
)

matches = []
previous_start = 0
previous_end = 0

for index, response_path in enumerate(response_files):

    data = json.loads(
        response_path.read_text(encoding="utf-8")
    )

    transcription = " ".join(
        segment["text"].strip()
        for segment in data["segments"]
    )

    spoken_text = normalize(transcription)
    word_count = len(spoken_text.split())

    if word_count == 0:
        print("Empty transcription:", response_path.name)
        continue

    # Allow different book passage lengths
    window_sizes = [
        max(1, int(word_count * factor))
        for factor in [0.8, 1.0, 1.2]
    ]

    # First section: search the entire book
    if index == 0:
        search_start = 0
        search_end = len(book_words) - 1

    # Other sections: search near previous match
    else:
        search_start = max(0, previous_start - 30)
        search_end = min(
            len(book_words) - 1,
            previous_end + word_count * 3
        )

    best_score = -1
    best_start = 0
    best_end = 0

    # Search in small steps
    step = max(1, word_count // 10)

    for start in range(search_start, search_end + 1, step):

        for size in window_sizes:

            end = min(start + size, len(book_words))

            candidate = " ".join(book_words[start:end])

            score = fuzz.ratio(spoken_text, candidate)

            if score > best_score:
                best_score = score
                best_start = start
                best_end = end

    # Refine the best position word by word
    for start in range(
        max(search_start, best_start - step),
        min(search_end, best_start + step) + 1
    ):
        for size in window_sizes:

            end = min(start + size, len(book_words))
            candidate = " ".join(book_words[start:end])
            score = fuzz.ratio(spoken_text, candidate)

            if score > best_score:
                best_score = score
                best_start = start
                best_end = end

    previous_start = best_start
    previous_end = best_end

    matched_text = " ".join(
        book_words[best_start:best_end]
    )

    matches.append({
        "section": response_path.stem,
        "start_word": best_start,
        "end_word": best_end,
        "similarity": round(best_score, 2),
        "transcription": transcription,
        "book_text": matched_text,
        "needs_review": best_score < 85
    })

    print(
        response_path.stem,
        "| Score:", round(best_score, 2),
        "| Words:", best_start, "-", best_end
    )

# Save matching results
output_path = Path("../../data/test/book_matches_draft.json")

output_path.write_text(
    json.dumps(matches, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("\nSaved:", output_path)

section_000 | Score: 82.35 | Words: 0 - 51
section_001 | Score: 98.65 | Words: 51 - 151
section_002 | Score: 95.03 | Words: 150 - 237
section_003 | Score: 98.26 | Words: 240 - 330
section_004 | Score: 96.54 | Words: 331 - 452
section_005 | Score: 98.93 | Words: 453 - 550
section_006 | Score: 97.39 | Words: 550 - 642
section_007 | Score: 98.91 | Words: 640 - 738
section_008 | Score: 93.84 | Words: 738 - 823
section_009 | Score: 94.55 | Words: 828 - 890

Saved: ../../data/test/book_matches_draft.json


In [6]:

metadata_path = Path("temporary_sections.json")

metadata = json.loads(
    metadata_path.read_text(encoding="utf-8")
)

print("Metadata type:", type(metadata).__name__)

if isinstance(metadata, dict):
    print("Keys:", list(metadata.keys()))
    print("Example:", next(iter(metadata.items()), None))
else:
    print("Number of sections:", len(metadata))
    print("First section:", metadata[0] if metadata else None)


Metadata type: list
Number of sections: 10
First section: {'filename': 'section_000.wav', 'start': 0, 'end': 59.9}


In [7]:

import subprocess

# Load our saved temporary section metadata
sections = json.loads(
    Path("temporary_sections.json").read_text(encoding="utf-8")
)

# Sort sections by their starting time
sections = sorted(sections, key=lambda s: float(s["start"]))

# Find the original audio duration
audio_path = Path("../../data/test/test_audio.mp3")

duration = float(
    subprocess.check_output(
        [
            "ffprobe", "-v", "error",
            "-show_entries", "format=duration",
            "-of", "csv=p=0",
            str(audio_path)
        ],
        text=True
    ).strip()
)

# Compare neighboring sections
differences = [
    float(b["start"]) - float(a["end"])
    for a, b in zip(sections, sections[1:])
]

print(f"Original duration: {duration:.2f} seconds")
print(f"Total sections: {len(sections)}")
print(f"First section starts: {sections[0]['start']}s")
print(f"Last section ends: {max(float(s['end']) for s in sections):.2f}s")
print(f"Gaps over 0.1s: {sum(d > 0.1 for d in differences)}")
print(f"Overlaps over 0.1s: {sum(d < -0.1 for d in differences)}")


Original duration: 591.73 seconds
Total sections: 10
First section starts: 0s
Last section ends: 591.73s
Gaps over 0.1s: 0
Overlaps over 0.1s: 9


In [8]:

# Our existing folders
audio_dir = Path("temporary_sections")
draft_dir = Path("../../data/test/alignment_drafts")

# Find files belonging to section_000
audio_files = sorted(audio_dir.rglob("*section_000*"))
draft_files = sorted(draft_dir.rglob("*section_000*"))

print("Audio files:")
for file in audio_files:
    print(file.resolve())

print("\nAlignment draft files:")
for file in draft_files:
    print(file.resolve())

# Preview the draft if available
if draft_files:
    print("\nDraft preview:")
    print(draft_files[0].read_text(encoding="utf-8")[:500])


Audio files:
/home/intern/book_audio_alignment/src/notebooks/temporary_sections/section_000.wav

Alignment draft files:


In [9]:

project_root = Path("/home/intern/book_audio_alignment")

# Search for files related to matching and alignment
keywords = ["alignment", "match", "draft", "section_000"]

for folder in ["data/test", "src/notebooks"]:
    folder_path = project_root / folder

    print(f"\nSearching: {folder}")

    matches = [
        file for file in folder_path.rglob("*")
        if file.is_file()
        and any(word in file.name.lower() for word in keywords)
        and file.suffix.lower() in [".json", ".jsonl", ".txt"]
    ]

    for file in sorted(matches)[:40]:
        print(file.relative_to(project_root))

    print(f"Total matching files: {len(matches)}")



Searching: data/test
data/test/book_matches_draft.json
data/test/matched_audio_text.txt
data/test/matched_book_draft.txt
Total matching files: 3

Searching: src/notebooks
Total matching files: 0


In [10]:

# Locate our matching files
data_dir = project_root / "data/test"

files_to_check = [
    "book_matches_draft.json",
    "matched_audio_text.txt",
    "matched_book_draft.txt"
]

for filename in files_to_check:
    file_path = data_dir / filename

    print(f"\n{'=' * 40}")
    print(f"FILE: {filename}")

    if not file_path.exists():
        print("File not found")
        continue

    if file_path.suffix == ".json":
        data = json.loads(file_path.read_text(encoding="utf-8"))

        print("Data type:", type(data).__name__)

        if isinstance(data, dict):
            print("Keys:", list(data.keys()))
            print("Sample:", str(list(data.items())[:2])[:800])

        elif isinstance(data, list):
            print("Number of items:", len(data))
            print("First item:", str(data[0])[:800] if data else "Empty")

    else:
        content = file_path.read_text(encoding="utf-8")
        print("Total characters:", len(content))
        print("Preview:", content[:500])



FILE: book_matches_draft.json
Data type: list
Number of items: 10
First item: {'section': 'section_000', 'start_word': 0, 'end_word': 51, 'similarity': 82.35, 'transcription': 'Qadrli bolajonlar, o\'zbek xalq ertaklari asosidagi baxt daraxti nomli ertakni aytib beraman. Bolalar adabiyoti jurnali. Qadim zamonda bir podsho o\'tgan ekan, U taxtda doim g\'amgin o\'tirar ekan. Kunlarning birida o\'ng qo\'li vaziri undan: "Podshohim, nimagadir mudom g\'amginsiz, boisi nima?" deb so\'rabdi. "Mol-u davlatim yetarli, ammo o\'lgandan keyin bel bog\'lab bo\'zlaydigan o\'g\'lim yo\'q, o\'g\'il ko\'rsam, boshqa armonim qolmaydi", debdi podsho. "Olampanoh, sizga maslahatim bor.', 'book_text': "ain ochilar y adim zamonda bir podsho o'tgan ekan u taxtda doim g'amgin o'tirarkan kunlarning birida o'ng 401 vaziri undan podshohim nimagadir mudom фатат 512 boisi nima deb so'rabdi mol u 

FILE: matched_audio_text.txt
Total characters: 325
Preview: bir podsho o'tgan ekan u taxtda doim g'amgin o'tirar ekan k

In [11]:

# Our audio file
audio_path = (
    project_root
    / "src/notebooks/temporary_sections/section_000.wav"
)

# Make sure the audio exists
assert audio_path.exists(), "Audio file not found!"

# Where we will save our manifest
manifest_path = (
    project_root / "data/test/nemo_manifest_section_000.jsonl"
)

# Create one manifest entry
entry = {
    "audio_filepath": str(audio_path.resolve())
}

# Save it as JSON Lines
manifest_path.write_text(
    json.dumps(entry, ensure_ascii=False) + "\n",
    encoding="utf-8"
)

print("Manifest created:", manifest_path)
print("Contents:")
print(manifest_path.read_text())


Manifest created: /home/intern/book_audio_alignment/data/test/nemo_manifest_section_000.jsonl
Contents:
{"audio_filepath": "/home/intern/book_audio_alignment/src/notebooks/temporary_sections/section_000.wav"}



In [12]:

import importlib.util
import sys

print("Python version:", sys.version.split()[0])

# Check whether the required libraries are installed
for package in ["torch", "nemo"]:
    installed = importlib.util.find_spec(package) is not None
    print(f"{package} installed: {installed}")

# Check whether PyTorch can access an NVIDIA GPU
if importlib.util.find_spec("torch") is not None:
    import torch

    print("PyTorch version:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())

    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))

# Confirm our manifest exists
manifest_file = (
    project_root / "data/test/nemo_manifest_section_000.jsonl"
)

print("Manifest exists:", manifest_file.exists())


Python version: 3.12.16
torch installed: True
nemo installed: False
PyTorch version: 2.9.1+cpu
CUDA available: False
Manifest exists: True


In [13]:

import importlib.util
import sys

print("Python version:", sys.version.split()[0])

# Check whether the required libraries are installed
for package in ["torch", "nemo"]:
    installed = importlib.util.find_spec(package) is not None
    print(f"{package} installed: {installed}")

# Check whether PyTorch can access an NVIDIA GPU
if importlib.util.find_spec("torch") is not None:
    import torch

    print("PyTorch version:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())

    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))

# Confirm our manifest exists
manifest_file = (
    project_root / "data/test/nemo_manifest_section_000.jsonl"
)

print("Manifest exists:", manifest_file.exists())


Python version: 3.12.16
torch installed: True
nemo installed: False
PyTorch version: 2.9.1+cpu
CUDA available: False
Manifest exists: True


In [14]:

import shutil
import subprocess

# Check if the NVIDIA diagnostic tool exists
nvidia_smi = shutil.which("nvidia-smi")

if nvidia_smi:
    result = subprocess.run(
        [nvidia_smi],
        capture_output=True,
        text=True
    )
    print(result.stdout or result.stderr)
else:
    print("nvidia-smi not found")


Sat Nov  7 18:02:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 590.48.01              Driver Version: 590.48.01      CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GB10                    On  |   0000000F:01:00.0  On |                  N/A |
| N/A   83C    P0             49W /  N/A  | Not Supported          |     93%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [15]:

import platform

print("Architecture:", platform.machine())
print("Python executable:", sys.executable)
print("PyTorch version:", torch.__version__)
print("PyTorch CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())



Architecture: aarch64
Python executable: /home/intern/book_audio_alignment/.venv/bin/python
PyTorch version: 2.9.1+cpu
PyTorch CUDA build: None
CUDA available: False
